<div style="border:solid orange 2px; padding: 20px">

Hi Yaser!

My name is <b><a href="https://hub.tripleten.com/u/4e3b1e8f">Yuliya Nikishina</a></b>, and I'll be reviewing your project today.

Great work on the machine learning part of the project! You successfully built and compared several classification models, tuned hyperparameters, selected the best model using the validation set, achieved the required accuracy on the test set, and completed the optional sanity check.

You can find my comments throughout the notebook. I use the following color coding:

<div class="alert alert-success" style="border-left: 7px solid green">
<b>✅ Great Job</b> – Something was completed correctly or particularly well.
</div>

<div class="alert alert-warning" style="border-left: 7px solid orange">
<b>⚠️ Recommendation</b> – Suggestions for improvement or best practices. These comments do not prevent project approval.
</div>

<div class="alert alert-danger" style="border-left: 7px solid red">
<b>⛔️ Needs Fixing</b> – These issues must be addressed before the project can be approved.
</div>

Please **do not delete or modify my comments** when working on the revision. If you'd like to reply or ask a question, feel free to add your response directly below my comment in a separate Markdown cell. For example:

<div class="alert alert-info" style="border-left: 7px solid #17a2b8">
<b>💬 Student's reply</b><br>
Your response here.
</div>

---

<b>Project review result:</b>

The machine learning workflow itself is largely correct, and your final model successfully exceeds the required accuracy threshold. However, the notebook needs some revision before it can be approved.

The main issue is the notebook structure: the entire project is currently placed in a single code cell, including all explanations and conclusions. Please divide the work into logical code cells and use Markdown cells for your analysis and conclusions. In addition, please correct the dataset path so the submitted notebook can run in the TripleTen environment without any changes from the reviewer - please see my detailed comments below the code for more information.

Once these items are fixed, the project should be in good shape for the next review. 🚀

</div>

<div style="border:solid green 2px; padding: 20px; border-radius: 10px">
<b>Reviewer's comment v2</b>

<b>Overall Feedback</b>

Hello <b>Yaser</b>!

My name is <b>Carlos Huapaya</b> and I'm back for this second iteration review.

You can find me on the HUB as https://hub.tripleten.com/u/6068137a if you need further feedback or if you have any questions at all.

Thank you for resubmitting and addressing the previous feedback! 🙌 The improvements you made are clear and the project is now in great shape.

---

You'll find specific feedback in the notebook in the **Reviewer's comment v2** sections.

**✅ What's Working Well**
- Both previous blocking issues have been resolved: the notebook now runs end-to-end successfully, and the dataset path is correct (`/datasets/users_behavior.csv` loads without errors).
- Clean 60/20/20 three-way split is implemented correctly using two calls to 'train_test_split', with no data leakage.
- Thorough hyperparameter tuning across three model types (Decision Tree, Random Forest, Logistic Regression) using only the validation set — exactly the right approach.
- The final model achieves ~0.80 test accuracy, comfortably above the 0.75 threshold, and the sanity check against the DummyClassifier is a nice touch that confirms genuine learning.

**🚧 Suggested Tweaks & Areas for Attention**
- The final model is retrained on the training set only before evaluating on the test set. A small optional improvement would be to retrain on train + validation combined before the final test evaluation, which is a common production practice — but this is not required here.
- The comment in Cell 9 references the summary table as approximate values (e.g., '~0.79') — it would be slightly more precise to print the exact best scores, but this is cosmetic only.

Overall, this is a well-structured, clean, and complete project. Both previously flagged red issues are now fixed, the code runs top-to-bottom, the methodology is sound, and the conclusions are well-articulated. **This project is approved!** Great work, Yaser — keep up this momentum as you move into more advanced topics! 🚀

---

  <p style="margin-top: 12px"><b>How to read my comments</b></p>

  <div class="alert alert-success"><b>✅ Success:</b> correct and well done</div>
  <div class="alert alert-warning"><b>💡 Recommendation:</b> works, but could be clearer / stronger / more efficient</div>
  <div class="alert alert-danger"><b>❗ Needs Fix:</b> affects correctness — must be fixed for approval</div>

  <p style="margin-top: 10px">
  <b>Please don't move, edit, or delete my comments.</b>
  If you have any comments or questions, I'd be happy to read them within these boxes:
  </p>

  <div style="background:#90d5ec; padding: 10px 12px; border-radius: 8px; margin-top: 10px">
    <b>🗣 Student notes:</b> Student's comments with thoughts, questions, or justification.
  </div>
</div>

In [1]:
# ============================================================
# Megaline Plan Recommendation Model
# ============================================================
# Goal: Megaline wants to recommend one of its newer plans (Smart or Ultra)
# to subscribers still on legacy plans. Using behavior data from subscribers
# who already switched, we build a classification model that predicts which
# plan (is_ultra) fits a subscriber based on their monthly calls, minutes,
# messages, and internet usage.
#
# Target accuracy: 0.75 on the test set.
#
# Plan of attack:
#   1. Load and explore the data
#   2. Split into train / validation / test sets
#   3. Try a few models and hyperparameters, compare accuracy on validation
#   4. Evaluate the best model on the test set
#   5. Sanity-check the model against a simple baseline
# ============================================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score


In [2]:
# ------------------------------------------------------------
# 1. Open and explore the data
# ------------------------------------------------------------
df = pd.read_csv('/datasets/users_behavior.csv')
print(df.head())
print()

df.info()  # No missing values, all columns numeric - matches the data description
print()


   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB



In [3]:
# Since preprocessing was already done in a previous project, just double-check
# there's nothing off (missing values, duplicates) before moving on.
print('Duplicate rows:', df.duplicated().sum())
print()
print(df.describe())
print()

# Check class balance
print(df['is_ultra'].value_counts(normalize=True))
# About 69% of users are on Smart (0) and 31% are on Ultra (1).
# This is a moderately imbalanced dataset - a model that always predicts
# "Smart" would already get ~69% accuracy. Useful as a sanity-check baseline later.
print()


Duplicate rows: 0

             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246     0.461100
min       0.000000     0.000000     0.000000      0.000000     0.000000
25%      40.000000   274.575000     9.000000  12491.902500     0.000000
50%      62.000000   430.600000    30.000000  16943.235000     0.000000
75%      82.000000   571.927500    57.000000  21424.700000     1.000000
max     244.000000  1632.060000   224.000000  49745.730000     1.000000

0    0.693528
1    0.306472
Name: is_ultra, dtype: float64



In [4]:
# ------------------------------------------------------------
# 2. Split the data into train, validation, and test sets
# ------------------------------------------------------------
# The task doesn't give us a separate test set, so we split the source data
# ourselves. Since there's no dedicated validation set either, we use a common
# 3-way split: 60% train / 20% validation / 20% test.
#   - Train:      used to fit the models
#   - Validation: used to compare different models/hyperparameters and pick the best
#   - Test:       used only once at the end, for an honest estimate of performance
#                 on unseen data
# This is done with two calls to train_test_split: first split off 60% train /
# 40% "temp", then split that 40% evenly into validation and test (20%/20% overall).

features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# First split: 60% train, 40% temp
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=12345)

# Second split: split the 40% temp into 20% validation, 20% test
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=12345)

print('Train:', features_train.shape)
print('Validation:', features_valid.shape)
print('Test:', features_test.shape)
print()


Train: (1928, 4)
Validation: (643, 4)
Test: (643, 4)



In [5]:
# ------------------------------------------------------------
# 3. Investigate different models and hyperparameters
# ------------------------------------------------------------
# Three model types make sense for this binary classification problem:
#   - Decision Tree: simple, easy to interpret, but can overfit if too deep
#   - Random Forest: an ensemble of trees, usually more accurate/stable
#   - Logistic Regression: a simple linear baseline model
# For each, train on the training set and check accuracy on the validation set,
# trying different hyperparameters to see what works best.

# --- 3.1 Decision Tree: tuning max_depth ---
best_tree_score = 0
best_tree_depth = 0


In [6]:
for depth in range(1, 11):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions = model.predict(features_valid)
    score = accuracy_score(target_valid, predictions)
    print(f'max_depth={depth}: validation accuracy = {score:.4f}')
    if score > best_tree_score:
        best_tree_score = score
        best_tree_depth = depth

print()
print(f'Best decision tree: max_depth={best_tree_depth}, accuracy={best_tree_score:.4f}')
# Accuracy improves as the tree gets a bit deeper, then drops off - a sign of
# overfitting once the tree gets too complex and starts memorizing the training
# data instead of generalizing.
print()


max_depth=1: validation accuracy = 0.7543
max_depth=2: validation accuracy = 0.7823
max_depth=3: validation accuracy = 0.7854
max_depth=4: validation accuracy = 0.7792
max_depth=5: validation accuracy = 0.7792
max_depth=6: validation accuracy = 0.7838
max_depth=7: validation accuracy = 0.7823
max_depth=8: validation accuracy = 0.7792
max_depth=9: validation accuracy = 0.7823
max_depth=10: validation accuracy = 0.7745

Best decision tree: max_depth=3, accuracy=0.7854



In [7]:
# --- 3.2 Random Forest: tuning n_estimators and max_depth ---
best_forest_score = 0
best_n_estimators = 0
best_forest_depth = 0

for n_estimators in range(10, 101, 10):
    for depth in range(1, 11):
        model = RandomForestClassifier(random_state=12345, n_estimators=n_estimators, max_depth=depth)
        model.fit(features_train, target_train)
        predictions = model.predict(features_valid)
        score = accuracy_score(target_valid, predictions)
        if score > best_forest_score:
            best_forest_score = score
            best_n_estimators = n_estimators
            best_forest_depth = depth

print(f'Best random forest: n_estimators={best_n_estimators}, max_depth={best_forest_depth}, accuracy={best_forest_score:.4f}')
# The random forest outperforms the single decision tree - averaging many trees
# together tends to reduce overfitting and gives more stable predictions.
print()


Best random forest: n_estimators=40, max_depth=8, accuracy=0.8087



In [8]:
# --- 3.3 Logistic Regression: baseline linear model ---
model_lr = LogisticRegression(random_state=12345, solver='liblinear')
model_lr.fit(features_train, target_train)
predictions = model_lr.predict(features_valid)
lr_score = accuracy_score(target_valid, predictions)

print(f'Logistic regression validation accuracy: {lr_score:.4f}')
# Logistic regression does the worst of the three. This suggests the relationship
# between usage behavior and plan choice isn't purely linear - tree-based models,
# which capture more complex patterns and interactions between features (e.g.
# "many minutes AND lots of data"), do better here.
print()


Logistic regression validation accuracy: 0.7092



In [9]:
# Findings so far:
#   Model                 | Best hyperparameters             | Validation accuracy
#   -----------------------------------------------------------------------------
#   Decision Tree          | tuned max_depth                  | ~0.79
#   Random Forest           | tuned n_estimators, max_depth   | ~0.81
#   Logistic Regression     | default                          | ~0.71
# The Random Forest gives the best validation accuracy, so we use it as the
# final model.


In [10]:
# ------------------------------------------------------------
# 4. Check the final model on the test set
# ------------------------------------------------------------
# Retrain the best random forest (best hyperparameters found above) and
# evaluate it ONCE on the held-out test set, which wasn't used for training
# or tuning.
final_model = RandomForestClassifier(
    random_state=12345,
    n_estimators=best_n_estimators,
    max_depth=best_forest_depth
)
final_model.fit(features_train, target_train)

test_predictions = final_model.predict(features_test)
test_accuracy = accuracy_score(target_test, test_predictions)

print(f'Test accuracy: {test_accuracy:.4f}')
# Test accuracy is above the 0.75 threshold required for the project, and it's
# close to the validation accuracy - a good sign the model isn't overfitting
# to the validation set.
print()


Test accuracy: 0.7963



In [11]:
# ------------------------------------------------------------
# 5. Sanity check
# ------------------------------------------------------------
# A tuned model should clearly beat a "dumb" model that ignores the features
# entirely. Compare against a DummyClassifier that always predicts the most
# frequent class (Smart, is_ultra=0).
dummy_model = DummyClassifier(strategy='most_frequent', random_state=12345)
dummy_model.fit(features_train, target_train)
dummy_predictions = dummy_model.predict(features_test)
dummy_accuracy = accuracy_score(target_test, dummy_predictions)

print(f'Dummy (most frequent class) accuracy: {dummy_accuracy:.4f}')
print(f'Random forest accuracy:                {test_accuracy:.4f}')
# The random forest beats the "always predict Smart" baseline by a solid margin,
# confirming it's actually learning something useful from the calls/minutes/
# messages/mb_used features rather than just exploiting the class imbalance.

Dummy (most frequent class) accuracy: 0.6843
Random forest accuracy:                0.7963


In [12]:
# ------------------------------------------------------------
# Conclusion
# ------------------------------------------------------------
# - The data was clean (no missing values or duplicates), so no extra
#   preprocessing was needed.
# - Split the data 60/20/20 into train/validation/test sets.
# - Compared a Decision Tree, a Random Forest, and Logistic Regression,
#   tuning hyperparameters on the validation set.
# - Random Forest (n_estimators and max_depth tuned) performed best on
#   validation and was selected as the final model.
# - On the test set, the final model reached ~0.80 accuracy, comfortably
#   above the 0.75 target.
# - A sanity check against a dummy "most frequent class" baseline (~0.68
#   accuracy) confirms the model is genuinely learning from user behavior,
#   not just guessing the majority class.


<div class="alert alert-danger"; style="border-left: 7px solid red">
<b>⛔️ Reviewer's comment, v. 1</b>

At the moment, the entire project — data loading, exploration, data splitting, model training, evaluation, and conclusions — is placed in a single code cell, with all explanations written as Python comments. Please reorganize the notebook before resubmitting it. 

A Jupyter Notebook should be divided into logical sections. Please use separate **code cells** for the main steps of the analysis and **Markdown cells** for explanations, findings, and conclusions. For example, you can organize the project into:

* Data overview
* Train / validation / test split
* Decision Tree
* Random Forest
* Logistic Regression
* Test evaluation
* Sanity check
* Conclusion

This will make the notebook much easier to read, run, debug, and review.
</div>

<div class="alert alert-danger"; style="border-left: 7px solid red">
<b>⛔️ Reviewer's comment, v. 1</b>

Please check the dataset path before resubmitting the project. In the submitted version, the path did not work in the TripleTen environment, so I had to correct it in order to run the notebook.

Please use:

<code>/datasets/users_behavior.csv</code>

and make sure the notebook runs successfully from beginning to end on the platform without requiring any manual changes from the reviewer.
</div>

<div class="alert alert-success">
<b>Reviewer's comment v2:</b>

The notebook now runs completely end-to-end — all 12 cells executed successfully. This was the primary blocker from the previous iteration and it is fully resolved. Great job getting everything working! ✅
</div>